In [16]:
## Import libraries
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Load Dataset

In [17]:
df = pd.read_csv("../data/online vs store shopping dataset.csv")
df.head()

,age,monthly_income,daily_internet_hours,smartphone_usage_years,social_media_hours,online_payment_trust_score,tech_savvy_score,monthly_online_orders,monthly_store_visits,avg_online_spend,...,free_return_importance,product_availability_online,impulse_buying_score,need_touch_feel_score,brand_loyalty_score,environmental_awareness,time_pressure_level,gender,city_tier,shopping_preference
0,56,221111,6.5,12,0.7,1,6,16,16,28551,...,7,7,1,3,6,5,2,Other,Tier 3,Store
1,69,96029,8.2,13,2.7,6,9,14,1,124056,...,3,4,9,6,8,1,7,Male,Tier 3,Hybrid
2,46,19055,6.4,4,2.1,10,8,2,0,81939,...,4,10,1,1,3,3,3,Female,Tier 3,Store
3,32,53170,6.4,11,0.7,2,10,20,3,35901,...,10,2,4,8,2,6,6,Female,Tier 1,Store
4,60,244016,6.0,5,0.7,2,5,18,16,131971,...,2,5,8,9,7,1,6,Male,Tier 3,Store


## Define Features and Target

In [18]:
X = df.drop(columns=["shopping_preference"])
y = df["shopping_preference"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11789, 24)
y shape: (11789,)


## Encode Target Variable

In [20]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_)
print("First 5 encoded values:", y_encoded[:5])

Classes: ['Hybrid' 'Online' 'Store']
First 5 encoded values: [2 0 2 2 2]


## Data Quality Check

In [21]:
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

Number of duplicate rows: 0


## Outlier Handling (IQR Method)

In [22]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
print(num_cols)

Index(['age', 'monthly_income', 'daily_internet_hours',
       'smartphone_usage_years', 'social_media_hours',
       'online_payment_trust_score', 'tech_savvy_score',
       'monthly_online_orders', 'monthly_store_visits', 'avg_online_spend',
       'avg_store_spend', 'discount_sensitivity', 'return_frequency',
       'avg_delivery_days', 'delivery_fee_sensitivity',
       'free_return_importance', 'product_availability_online',
       'impulse_buying_score', 'need_touch_feel_score', 'brand_loyalty_score',
       'environmental_awareness', 'time_pressure_level'],
      dtype='str')


In [23]:
for col in num_cols:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((X[col] < lower) | (X[col] > upper)).sum()
    print(col, "->", outliers)

age -> 0
monthly_income -> 0
daily_internet_hours -> 25
smartphone_usage_years -> 0
social_media_hours -> 0
online_payment_trust_score -> 0
tech_savvy_score -> 0
monthly_online_orders -> 0
monthly_store_visits -> 0
avg_online_spend -> 0
avg_store_spend -> 0
discount_sensitivity -> 0
return_frequency -> 0
avg_delivery_days -> 0
delivery_fee_sensitivity -> 0
free_return_importance -> 0
product_availability_online -> 0
impulse_buying_score -> 0
need_touch_feel_score -> 0
brand_loyalty_score -> 0
environmental_awareness -> 0
time_pressure_level -> 0


In [24]:
X.describe()

,age,monthly_income,daily_internet_hours,smartphone_usage_years,social_media_hours,online_payment_trust_score,tech_savvy_score,monthly_online_orders,monthly_store_visits,avg_online_spend,...,return_frequency,avg_delivery_days,delivery_fee_sensitivity,free_return_importance,product_availability_online,impulse_buying_score,need_touch_feel_score,brand_loyalty_score,environmental_awareness,time_pressure_level
count,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,...,11789.00000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000,11789.000000
mean,48.729409,131704.282382,6.011367,7.597930,2.514471,5.498770,5.534312,24.677581,9.482144,74554.929341,...,4.46747,3.999661,5.468827,5.462041,5.518704,5.486386,5.485368,5.532021,5.448554,5.504114
std,17.899445,68120.726684,1.976811,4.011628,1.263047,2.880366,2.887251,14.431277,5.728825,43167.126595,...,2.88545,1.995881,2.870195,2.882177,2.867613,2.877918,2.877264,2.848796,2.872740,2.876561
min,18.000000,15005.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,523.000000,...,0.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,33.000000,72450.000000,4.600000,4.000000,1.600000,3.000000,3.000000,12.000000,5.000000,36797.000000,...,2.00000,2.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
50%,49.000000,131916.000000,6.000000,8.000000,2.500000,5.000000,6.000000,25.000000,9.000000,74859.000000,...,4.00000,4.000000,5.000000,5.000000,6.000000,5.000000,5.000000,6.000000,5.000000,6.000000
75%,64.000000,190505.000000,7.400000,11.000000,3.400000,8.000000,8.000000,37.000000,14.000000,112134.000000,...,7.00000,6.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000
max,79.000000,249989.000000,12.000000,14.000000,6.000000,10.000000,10.000000,49.000000,19.000000,149996.000000,...,9.00000,7.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000


## Encode Categorical Features

In [25]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns
print("Categorical columns:", list(cat_cols))

Categorical columns: ['gender', 'city_tier']


In [26]:
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=False)

In [27]:
print("Original X shape:", X.shape)
print("Encoded X shape:", X_encoded.shape)
X_encoded.head()

Original X shape: (11789, 24)
Encoded X shape: (11789, 28)


,age,monthly_income,daily_internet_hours,smartphone_usage_years,social_media_hours,online_payment_trust_score,tech_savvy_score,monthly_online_orders,monthly_store_visits,avg_online_spend,...,need_touch_feel_score,brand_loyalty_score,environmental_awareness,time_pressure_level,gender_Female,gender_Male,gender_Other,city_tier_Tier 1,city_tier_Tier 2,city_tier_Tier 3
0,56,221111,6.5,12,0.7,1,6,16,16,28551,...,3,6,5,2,False,False,True,False,False,True
1,69,96029,8.2,13,2.7,6,9,14,1,124056,...,6,8,1,7,False,True,False,False,False,True
2,46,19055,6.4,4,2.1,10,8,2,0,81939,...,1,3,3,3,True,False,False,False,False,True
3,32,53170,6.4,11,0.7,2,10,20,3,35901,...,8,2,6,6,True,False,False,True,False,False
4,60,244016,6.0,5,0.7,2,5,18,16,131971,...,9,7,1,6,False,True,False,False,False,True


## Train-Test Split

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (9431, 28)
X_test shape: (2358, 28)
y_train shape: (9431,)
y_test shape: (2358,)


## Feature Scaling

In [30]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [31]:
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (9431, 28)
X_test_scaled shape: (2358, 28)


In [32]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

X_train_scaled.head()

,age,monthly_income,daily_internet_hours,smartphone_usage_years,social_media_hours,online_payment_trust_score,tech_savvy_score,monthly_online_orders,monthly_store_visits,avg_online_spend,...,need_touch_feel_score,brand_loyalty_score,environmental_awareness,time_pressure_level,gender_Female,gender_Male,gender_Other,city_tier_Tier 1,city_tier_Tier 2,city_tier_Tier 3
7386,0.571821,0.020875,-0.101470,-0.640674,-1.988943,-1.215427,-1.222852,-0.399099,1.321018,1.398050,...,0.870846,-0.888130,0.887083,1.563913,-0.708851,-0.713584,1.430794,-0.708851,-0.701940,1.407381
7066,0.013047,1.496722,1.115269,-0.640674,-0.565759,1.219225,-0.532559,-0.330008,-0.254791,-1.024133,...,0.522678,1.220885,1.235119,-1.559314,1.410734,-0.713584,-0.698912,1.410734,-0.701940,-0.710540
11397,1.186473,-0.180288,0.963177,1.352875,-0.407628,0.175803,-0.532559,-0.053641,0.970838,-0.886294,...,-0.521829,1.572387,-0.853095,0.175812,-0.708851,-0.713584,1.430794,-0.708851,-0.701940,1.407381
8229,0.627699,-0.548004,-0.202865,0.356100,-1.198285,-1.563234,1.193172,1.051826,-0.604971,-0.653276,...,1.219015,-0.888130,-0.853095,0.175812,1.410734,-0.713584,-0.698912,-0.708851,1.424624,-0.710540
2124,-1.607400,1.362228,-2.027974,-1.139061,-1.435483,-0.519812,0.157733,1.397284,-0.955151,1.619214,...,0.522678,0.517880,-0.853095,0.522838,-0.708851,-0.713584,1.430794,-0.708851,1.424624,-0.710540
